# 34 — BETO con preentrenamiento adaptado al dominio (Fase X)

**Idea propia.** BETO se preentrenó con texto en español "general" (Wikipedia, noticias, libros). Los títulos de MercadoLibre son otro idioma: "Remera Talle S Nva Sin Uso Envio Gratis", códigos de modelo, abreviaturas. Antes del fine-tuning de siempre, se lo pone a **leer títulos de MercadoLibre** con la misma tarea con la que se preentrenó.

**Qué hace, en simple** (`notebooks/bert_dapt.py`):
1. **Paso 1, MLM (*masked language modeling*):** se toman los 100.000 textos "título [SEP] garantía" de train + test, se tapa al azar el 15% de los tokens y BETO aprende a **adivinar la palabra tapada** mirando el resto. Ejemplo: "Remera [MASK] S Sin Uso" → "Talle". No hay `new`/`used` en ningún lado. 1 época (~23 min).
2. **Paso 2, fine-tuning:** exactamente el de la Fase Q/T (2 épocas, lr 2e-5, …), con los folds de la semilla 42, pero partiendo del BETO del paso 1 en lugar del original. 5 globales `sin_k` (~21 min cada uno).
3. **Comparación:** la columna de texto pasa a ser (logística + BETO adaptado) / 2, en lugar de (logística + BETO) / 2.

**Por qué no es leakage:**
- El paso 1 **no ve ninguna etiqueta**: solo lee texto (el script ni siquiera lee la columna `condition`).
- Que lea los títulos del test y de la validación es **transductivo**, como los conteos sobre train + test (Fases B, P y R): en el uso real, los títulos a predecir están disponibles antes de predecir.
- Lo que aprende es vocabulario ("Nva" ≈ "nueva", "Talle" va con un número), no qué publicación es `new`.
- Las etiquetas entran recién en el paso 2, con el cross-fitting de siempre: cada probabilidad sale de un modelo que no vio esa fila.

**⚠️ CV sesgada como en la Fase V** (solo los 5 globales): para el fold externo k, las filas de train del fold j reciben `sin_j`, que se entrenó con el fold k. Por eso la referencia es la **variante a sesgada igual** (BETO común con sus 5 globales, ya calculada en la 32), y la a exacta se muestra al lado. Para la versión exacta harían falta 10 ajustes "de a pares" más (~14 min cada uno, ~2,3 h).

**Material:** SIA `S12 - Redes Neuronales - NLP.pdf`, slides 34-35 (pre-entrenamiento como *transfer learning* para texto: el modelo aprende a predecir palabras con mucho texto y después se ajusta a la tarea). **No está en el material:** la tarea MLM de BERT, el preentrenamiento adaptado al dominio (*domain-adaptive pretraining*) y el fine-tuning (ver Fase Q).

In [1]:
import sys
sys.path.append("..")
import os, time, warnings
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold

from features import (aplanar_anidadas, agregar_features_contenido, agregar_categoricas_crudas, agregar_combos,
                      armar_pipeline_histgb_combos, armar_pipeline_final, INDICES_CAT)

pd.set_option("display.width", 220)
SEED = 42
carpeta_lab = ".cache_laboratorio"
crudo = pd.read_json("../data/train_data.jsonlines", lines=True)
y = (crudo["condition"] == "new").astype(int)
y_np = y.to_numpy()
n = len(crudo)
train = agregar_combos(agregar_categoricas_crudas(agregar_features_contenido(aplanar_anidadas(crudo))))
folds = np.empty(n, dtype=int)
for k, (_, v) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(train, y)):
    folds[v] = k + 1


def acc_por_fold(prob, filas=None):
    filas = np.ones(n, bool) if filas is None else filas
    acierto = ((np.asarray(prob) >= 0.5).astype(int) == y_np)
    return pd.Series(acierto[filas]).groupby(folds[filas]).mean().to_numpy()


def comparar(acc, ref):
    diferencias = np.round(acc - ref, 10)
    gana = int((diferencias > 0).sum())
    return {"media": acc.mean(), "dif_media": diferencias.mean(), "dif_desvio": diferencias.std(), "folds_gana": gana,
            "pasa": gana == 5 or (gana == 4 and diferencias.mean() > diferencias.std())}


sin_historial = np.zeros(n, bool)
for k in range(1, 6):
    va = folds == k
    sin_historial[va] = ~train.loc[va, "seller_id"].isin(set(train.loc[~va, "seller_id"])).to_numpy()


def globales(carpeta):
    p = np.full(n, np.nan)
    for k in range(1, 6):
        d = np.load(os.path.join(carpeta_lab, carpeta, f"sin_{k}.npz"))
        es_train = d["filas"] < n
        p[d["filas"][es_train]] = d["prob"][es_train]
    assert not np.isnan(p).any()
    return p


beto = globales("bert")
beto_dapt = globales("bert_dapt")
logistica = pd.read_csv(os.path.join(carpeta_lab, "oof_texto_logistica.csv"))["prob"].to_numpy()
logistica_fold = {k: np.load(os.path.join(carpeta_lab, "textos_31", f"logistica_fold{k}.npy")) for k in range(1, 6)}

# Referencias: variante a EXACTA (Fase T) y a SESGADA igual (5 globales de BETO, calculada en la 32)
oof_t = pd.read_csv(os.path.join(carpeta_lab, "oof_texto_combinado.csv"))
ref_hgb = oof_t["(a) promedio simple"].to_numpy()
ref_final = 0.5 * oof_t["catboost (a) promedio simple"].to_numpy() + 0.5 * ref_hgb
oof_32 = pd.read_csv(os.path.join(carpeta_lab, "oof_beto_contexto.csv"))
ref_sesg_hgb = oof_32["histgb | a sesgada (logística + BETO)"].to_numpy()
ref_sesg_final = 0.5 * oof_32["catboost | a sesgada (logística + BETO)"].to_numpy() + 0.5 * ref_sesg_hgb
print(f"a exacta: final {acc_por_fold(ref_final).mean():.4f} | a sesgada: final {acc_por_fold(ref_sesg_final).mean():.4f}")

a exacta: final 0.9435 | a sesgada: final 0.9432


## (a) BETO adaptado solo, contra BETO original (fold por fold)

OOF de los 5 globales: cada fila la predice el modelo que no la vio, así que esta comparación **no** tiene sesgo.

In [2]:
por_fold_a = pd.DataFrame({
    "acc BETO": acc_por_fold(beto), "acc BETO adaptado": acc_por_fold(beto_dapt),
    "logloss BETO": [log_loss(y_np[folds == k], beto[folds == k]) for k in range(1, 6)],
    "logloss BETO adaptado": [log_loss(y_np[folds == k], beto_dapt[folds == k]) for k in range(1, 6)],
}, index=pd.Index(range(1, 6), name="fold"))
por_fold_a["dif acc"] = por_fold_a["acc BETO adaptado"] - por_fold_a["acc BETO"]
por_fold_a.loc["media"] = por_fold_a.mean()
print("Sin historial — BETO:", acc_por_fold(beto, sin_historial).mean().round(4),
      "| adaptado:", acc_por_fold(beto_dapt, sin_historial).mean().round(4))
print("Correlación BETO vs adaptado:", np.corrcoef(beto, beto_dapt)[0, 1].round(3),
      "| logística + BETO:", acc_por_fold((logistica + beto) / 2).mean().round(4),
      "| logística + adaptado:", acc_por_fold((logistica + beto_dapt) / 2).mean().round(4))
por_fold_a.round(4)

Sin historial — BETO: 0.7926 | adaptado: 0.7995
Correlación BETO vs adaptado: 0.967 | logística + BETO: 0.8589 | logística + adaptado: 0.8629


,acc BETO,acc BETO adaptado,logloss BETO,logloss BETO adaptado,dif acc
fold,,,,,
1,0.8469,0.8563,0.3525,0.3415,0.0094
2,0.8506,0.8554,0.3454,0.3369,0.0049
3,0.8491,0.8536,0.3501,0.3437,0.0044
4,0.8503,0.8556,0.3432,0.3391,0.0054
5,0.8519,0.8576,0.3428,0.3354,0.0057
media,0.8497,0.8557,0.3468,0.3393,0.0060


## (b) Columna de texto (logística + BETO adaptado) / 2 en el HistGB y el promedio final

Mismo procedimiento que la 32: el transformer `texto` se reemplaza por `passthrough` de la columna (misma posición, `INDICES_CAT` no cambia). Logística exacta por fold; BETO adaptado = OOF de los globales (sesgado igual que la referencia "a sesgada").

In [3]:
def armar(modelo):
    pipeline = armar_pipeline_histgb_combos() if modelo == "histgb" else armar_pipeline_final()
    transformers = pipeline.named_steps["preprocesador"].transformers
    i = [nm for nm, _, _ in transformers].index("texto")
    transformers[i] = ("texto", "passthrough", ["t_texto"])
    return pipeline


def correr_fold(k, modelo, columna):
    entrena, valida = folds != k, folds == k
    X = train.assign(t_texto=columna)
    params = {"modelo__cat_features": INDICES_CAT} if modelo == "catboost" else {}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pipeline = armar(modelo).fit(X[entrena], y[entrena], **params)
    return k, pipeline.predict_proba(X[valida])[:, 1]


def oof(modelo, columna_de_fold):
    prob = np.empty(n)
    for k, p in Parallel(n_jobs=5)(delayed(correr_fold)(k, modelo, columna_de_fold(k)) for k in range(1, 6)):
        prob[folds == k] = p
    return prob


archivo_oof = os.path.join(carpeta_lab, "oof_beto_dapt.csv")
guardado = pd.read_csv(archivo_oof) if os.path.exists(archivo_oof) else pd.DataFrame(
    {"fold": folds, "y": y_np, "beto_dapt_global": beto_dapt})
for modelo in ["histgb", "catboost"]:
    col = f"{modelo} | logística + BETO adaptado"
    if col not in guardado:
        inicio = time.time()
        guardado[col] = oof(modelo, lambda k: (logistica_fold[k] + beto_dapt) / 2)
        guardado.to_csv(archivo_oof, index=False)
        print(f"{col:45s} {(time.time() - inicio) / 60:.1f} min", flush=True)

p_h = guardado["histgb | logística + BETO adaptado"].to_numpy()
p_f = 0.5 * guardado["catboost | logística + BETO adaptado"].to_numpy() + 0.5 * p_h
resultados = {
    "HistGB vs a sesgada": comparar(acc_por_fold(p_h), acc_por_fold(ref_sesg_hgb)),
    "HistGB vs a exacta": comparar(acc_por_fold(p_h), acc_por_fold(ref_hgb)),
    "final vs a sesgada (referencia de igual a igual)": comparar(acc_por_fold(p_f), acc_por_fold(ref_sesg_final)),
    "final vs a exacta (la entrega)": comparar(acc_por_fold(p_f), acc_por_fold(ref_final)),
}
display(pd.DataFrame(resultados).T.round(5))
print("Sin historial (final): a exacta", acc_por_fold(ref_final, sin_historial).mean().round(4),
      "| a sesgada", acc_por_fold(ref_sesg_final, sin_historial).mean().round(4),
      "| adaptado", acc_por_fold(p_f, sin_historial).mean().round(4))
por_fold_b = pd.DataFrame({"a exacta": acc_por_fold(ref_final), "a sesgada": acc_por_fold(ref_sesg_final),
                           "logística + BETO adaptado": acc_por_fold(p_f)}, index=pd.Index(range(1, 6), name="fold"))
por_fold_b["dif vs a sesgada"] = por_fold_b["logística + BETO adaptado"] - por_fold_b["a sesgada"]
por_fold_b["dif vs a exacta"] = por_fold_b["logística + BETO adaptado"] - por_fold_b["a exacta"]
por_fold_b.loc["media"] = por_fold_b.mean()
por_fold_b.round(5)

histgb | logística + BETO adaptado            0.5 min


catboost | logística + BETO adaptado          1.7 min


,media,dif_media,dif_desvio,folds_gana,pasa
HistGB vs a sesgada,0.942943,0.000529,0.000583,4,False
HistGB vs a exacta,0.942943,0.000429,0.000733,4,False
final vs a sesgada (referencia de igual a igual),0.943514,0.000343,0.000856,3,False
final vs a exacta (la entrega),0.943514,-0.000014,0.000792,2,False


Sin historial (final): a exacta 0.8962 | a sesgada 0.8952 | adaptado 0.8958


,a exacta,a sesgada,logística + BETO adaptado,dif vs a sesgada,dif vs a exacta
fold,,,,,
1,0.94321,0.94350,0.94407,0.00057,0.00086
2,0.94221,0.94264,0.94157,-0.00107,-0.00064
3,0.94421,0.94336,0.94329,-0.00007,-0.00093
4,0.94350,0.94307,0.94450,0.00143,0.00100
5,0.94450,0.94329,0.94414,0.00086,-0.00036
media,0.94353,0.94317,0.94351,0.00034,-0.00001
